# Continuing Dohnuts Training on Kaggle (CPU & GPU Supported)

This notebook continues training the **[PsiACE/Dohnuts-0.1.0-0.8B](https://huggingface.co/PsiACE/Dohnuts-0.1.0-0.8B)** decision model on Kaggle with **dual support for CPU dry-testing and GPU production training**.

### Workflow Strategy:
1. **Phase 1 (CPU Verification)**: Run a quick 10-step dry run on Kaggle's free CPU to verify data loading, tokenization, forward/backward passes, and checkpoint export without burning GPU quota.
2. **Phase 2 (GPU Production)**: Switch the Kaggle accelerator to **GPU T4 x 2** (or **P100**), enabling compiled `flash-linear-attention` kernels and full batch training.

### What is Dohnuts?
- Built on **Qwen/Qwen3.5-0.8B**.
- Direct System 1 decision-making: multiple questions scored in a single forward pass without token generation.
- Trainable rank-8 LoRA adapter + linear candidate scoring head optimized with joint **RLCD + cross-entropy**.
- Co-trains **MMLU (`cais/mmlu`)** with a lightweight **replay buffer** (BoolQ, AG News) to prevent catastrophic forgetting.

## 1. Environment & Device Detection

In [ ]:
import torch

IS_CUDA = torch.cuda.is_available()
DEVICE_NAME = torch.cuda.get_device_name(0) if IS_CUDA else "CPU"

print(f"Device: {DEVICE_NAME}")
if IS_CUDA:
    print(f"CUDA Memory: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
    !nvidia-smi
else:
    print("Running in CPU mode. Ideal for testing data pipelines and rapid verification before switching to GPU!")

## 2. Dependencies & Kernel Setup

- On **GPU**: compiles `flash-linear-attention` for fused RMSNorm and SwiGLU.
- On **CPU**: installs PyTorch, Transformers, PEFT, and Datasets, using native ops for testing.

In [ ]:
!pip uninstall -y torchao
!pip install -q -U "transformers==5.17.0" "peft>=0.21.0" datasets>=3.0 huggingface-hub safetensors accelerate pillow pyarrow scikit-learn

if IS_CUDA:
    print("Compiling flash-linear-attention for CUDA...")
    !pip install -q flash-linear-attention --no-build-isolation
    import fla
    print("SUCCESS: flash-linear-attention loaded for GPU.")
else:
    print("CPU mode: Skipping GPU-only FLA kernels. Standard PyTorch execution enabled.")

## 3. Clone / Import Dohnuts Source Code

In [ ]:
import os
import sys
from pathlib import Path

# 1. Reset working directory to /kaggle/working to avoid deleted-CWD errors
work_dir = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".").resolve().parent
os.chdir(str(work_dir))

# 2. Freshly clone from pverghese/dohnuts
!rm -rf dohnuts
!git clone https://github.com/pverghese/dohnuts.git dohnuts

# 3. Enter the repository
repo_path = (work_dir / "dohnuts").resolve()
os.chdir(str(repo_path))
sys.path.insert(0, str(repo_path / "src"))
print(f"Working in: {os.getcwd()}")

# 4. Install package metadata in editable mode
!pip install -q -e . --no-deps

import dohnuts
print(f"Imported dohnuts version: {dohnuts.__version__}")


## 4. Download Pinned Base Model and Checkpoint Weights

In [ ]:
from huggingface_hub import snapshot_download

base_model_dir = Path(".cache/models/Qwen3.5-0.8B").resolve()
base_revision = "2fc06364715b967f1860aea9cf38778875588b17"

if not (base_model_dir / "revision.txt").exists():
    print(f"Downloading Qwen/Qwen3.5-0.8B (revision {base_revision})...")
    snapshot_download("Qwen/Qwen3.5-0.8B", revision=base_revision, local_dir=str(base_model_dir))
    (base_model_dir / "revision.txt").write_text(base_revision + "\n")
print("Base model ready.")

init_checkpoint_dir = Path(".cache/checkpoints/Dohnuts-0.1.0-0.8B").resolve()
if not (init_checkpoint_dir / "dohnuts.json").exists():
    print("Downloading PsiACE/Dohnuts-0.1.0-0.8B starting checkpoint...")
    snapshot_download(
        "PsiACE/Dohnuts-0.1.0-0.8B",
        allow_patterns=["dohnuts.json", "adapter.safetensors", "LICENSE"],
        local_dir=str(init_checkpoint_dir),
    )
print("Initial checkpoint ready.")

## 5. Prepare Dataset: MMLU + Replay Buffer

Generates `train.jsonl`, `dev.jsonl`, `calibration.jsonl`, and `test.jsonl`.

In [ ]:
# Set smoke_test=True for fast verification (works great on CPU or GPU)
SMOKE_TEST = True

!python scripts/prepare_mmlu.py --output data/processed/mmlu_replay {("--smoke-test" if SMOKE_TEST else "")}

## 6. Training Configuration & Recipe Generation

Automatically selects batch size and update budget based on CPU vs GPU.

In [ ]:
import json
import shutil
from dohnuts.recipe import training_recipe
from dohnuts.rlcd import RLCDConfig

output_dir = Path("runs/mmlu_experiment").resolve()
shutil.rmtree(output_dir, ignore_errors=True)
output_dir.mkdir(parents=True, exist_ok=True)
seed = 42
seed_dir = output_dir / f"seed-{seed}"
seed_dir.mkdir(exist_ok=True)

if not IS_CUDA:
    # Fast CPU settings for smoke-testing the pipeline
    steps = 10
    batch_size = 2
    accumulation = 1
elif SMOKE_TEST:
    # Fast GPU smoke-test (~3 mins)
    steps = 60
    batch_size = 8
    accumulation = 4
else:
    # Full GPU production training
    steps = 1200
    batch_size = 8
    accumulation = 4

recipe = training_recipe(
    model=base_model_dir,
    data=Path("data/processed/mmlu_replay"),
    seed=seed,
    rlcd=RLCDConfig(sigma=0.3, ce_weight=1.0),
    steps=steps,
    batch_size=batch_size,
    accumulation=accumulation,
)

recipe_path = output_dir / f"recipe-seed-{seed}.json"
recipe_path.write_text(json.dumps(recipe, indent=2))
print(f"Recipe created for {DEVICE_NAME}: steps={steps}, batch_size={batch_size}, accumulation={accumulation}")

## 7. Run Continued Training (RLCD + LoRA)

Initializes weights directly from `PsiACE/Dohnuts-0.1.0-0.8B`.

In [ ]:
!python -m dohnuts.train train \
    --config {recipe_path} \
    --run {seed_dir} \
    --initialize-from {init_checkpoint_dir}

## 8. Calibrate, Evaluate & Export Checkpoint

In [ ]:
!python -m dohnuts.train evaluate \
    --config {recipe_path} \
    --run {seed_dir}

# Copy exported checkpoint to Kaggle working directory
!mkdir -p /kaggle/working/checkpoint
!cp -r {output_dir}/checkpoint/* /kaggle/working/checkpoint/
print("Checkpoint exported to /kaggle/working/checkpoint:")
!ls -lh /kaggle/working/checkpoint

## 9. Test Decisions with the Newly Trained Predictor

In [ ]:
from dohnuts.predictor import Predictor

# Load freshly exported checkpoint
model = Predictor.from_checkpoint("/kaggle/working/checkpoint")

# Test 1: MMLU Multiple Choice Question
mmlu_query = model.predict(
    {"state": "Subject: anatomy"},
    {
        "bone_q": {
            "type": "choice",
            "instructions": "Which of the following bones is part of the axial skeleton?",
            "criteria": ["Femur", "Clavicle", "Sternum", "Scapula"],
        }
    },
)

print("=== MMLU Anatomy Decision ===")
print("Predicted Choice:", mmlu_query["answers"]["bone_q"]["choice"])
print("Probabilities:", mmlu_query["answers"]["bone_q"]["probabilities"])

# Test 2: General System 1 Boolean Truth (Retained skill from replay buffer)
bool_query = model.predict(
    {"state": "The solar system formed approximately 4.6 billion years ago."},
    {
        "age_statement": {
            "type": "noul",
            "instructions": "Is the solar system older than 4 billion years?",
        }
    },
)

print("\n=== General Boolean Truth Decision ===")
print("Statement holds (truth):", bool_query["answers"]["age_statement"]["noul"])
print("Confidence:", bool_query["answers"]["age_statement"]["confidence"])

## 10. (Optional) Push Trained Checkpoint to Hugging Face Hub

In [ ]:
PUSH_TO_HUB = False  # Set to True when ready
TARGET_REPO_ID = "your-username/Dohnuts-MMLU-0.8B"

if PUSH_TO_HUB:
    from huggingface_hub import HfApi, create_repo
    hf_token = os.environ.get("HF_TOKEN", "")
    if not hf_token:
        raise ValueError("Please set HF_TOKEN environment variable or Kaggle secret.")
    
    api = HfApi(token=hf_token)
    create_repo(TARGET_REPO_ID, repo_type="model", exist_ok=True)
    api.upload_folder(
        folder_path="/kaggle/working/checkpoint",
        repo_id=TARGET_REPO_ID,
        repo_type="model",
    )
    print(f"Successfully published checkpoint to https://huggingface.co/{TARGET_REPO_ID}")
else:
    print("Skipping Hugging Face upload. Checkpoint is stored locally in /kaggle/working/checkpoint.")